# PISTES — Accuracy Analysis
## ML vs Physics Comparison
**IT22258908 | IT4010 Research 2026**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib
import math
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

In [ ]:
# Load training pairs with wind
try:
    df = pd.read_csv(
        '../data/processed/lsd_real_wind_final.csv'
    )
except Exception:
    df = pd.read_csv(
        '../data/processed/lsd_with_wind_real.csv'
    )

# Remove rows with missing wind
df = df.dropna(
    subset=['wind_speed', 'wind_direction',
            'delta_lat', 'delta_lon']
)

print(f'Training pairs loaded: {len(df)}')
print(f'Countries: {df["country"].nunique()}')
print(f'Movement range: '
      f'{df["movement_km"].min():.1f} - '
      f'{df["movement_km"].max():.1f} km')
print(f'Mean movement: {df["movement_km"].mean():.1f} km')

In [ ]:
# Load models
model_lat = joblib.load('../models/model_lat.pkl')
model_lon = joblib.load('../models/model_lon.pkl')
scaler    = joblib.load('../models/scaler.pkl')

FEATURES = [
    'wind_u', 'wind_v',
    'wind_speed', 'wind_direction',
    'humidity'
]

# Use only available features
FEATURES = [f for f in FEATURES if f in df.columns]

X        = df[FEATURES].fillna(0).values
X_scaled = scaler.transform(X)

ml_pred_lat = model_lat.predict(X_scaled)
ml_pred_lon = model_lon.predict(X_scaled)

# Convert to km
ml_dlat_km = ml_pred_lat * 111.0
ml_dlon_km = (
    ml_pred_lon * 111.0
    * np.cos(np.radians(df['lat_d'].values))
)

ml_distance     = np.sqrt(ml_dlat_km**2 + ml_dlon_km**2)
actual_distance = df['movement_km'].values

ml_mae  = np.mean(np.abs(ml_distance - actual_distance))
ml_rmse = np.sqrt(np.mean((ml_distance - actual_distance)**2))

print('ML MODEL RESULTS:')
print(f'  MAE:  {ml_mae:.2f} km')
print(f'  RMSE: {ml_rmse:.2f} km')
print(f'  Predictions range: '
      f'{ml_distance.min():.1f} - '
      f'{ml_distance.max():.1f} km')

In [ ]:
def physics_predict_distance(wind_speed, humidity):
    amplitude  = (1.0
                  + 0.65 * 0.05
                  + (humidity / 100) * 0.05)
    base_daily = 1.5
    wind_daily = wind_speed * 0.6
    return (base_daily + wind_daily) * amplitude * 7


physics_distance = df.apply(
    lambda r: physics_predict_distance(
        r['wind_speed'],
        r.get('humidity', 75)
    ),
    axis=1
).values

physics_mae  = np.mean(np.abs(physics_distance - actual_distance))
physics_rmse = np.sqrt(np.mean(
    (physics_distance - actual_distance)**2
))

print('PHYSICS MODEL RESULTS:')
print(f'  MAE:  {physics_mae:.2f} km')
print(f'  RMSE: {physics_rmse:.2f} km')
print(f'  Predictions range: '
      f'{physics_distance.min():.1f} - '
      f'{physics_distance.max():.1f} km')

In [ ]:
# Hybrid: weighted average
# Physics more trust = 0.70
# ML less trust      = 0.30
PHYSICS_WEIGHT = 0.70
ML_WEIGHT      = 0.30

hybrid_distance = (
    physics_distance * PHYSICS_WEIGHT
    + ml_distance    * ML_WEIGHT
)

hybrid_mae = np.mean(
    np.abs(hybrid_distance - actual_distance)
)

print('HYBRID MODEL RESULTS:')
print(f'  MAE:  {hybrid_mae:.2f} km')
print(f'  Weights: Physics={PHYSICS_WEIGHT} ML={ML_WEIGHT}')

In [ ]:
summary = pd.DataFrame({
    'Method': [
        'ML (Ridge Regression)',
        'Physics EVF',
        'Hybrid (70% Physics + 30% ML)'
    ],
    'MAE (km)': [
        round(ml_mae, 2),
        round(physics_mae, 2),
        round(hybrid_mae, 2)
    ],
    'RMSE (km)': [
        round(ml_rmse, 2),
        round(physics_rmse, 2),
        round(np.sqrt(np.mean(
            (hybrid_distance - actual_distance)**2
        )), 2)
    ],
    'Training Data': [
        '5,097 pairs required',
        'No training data needed',
        'Minimal data needed'
    ],
    'Basis': [
        'Statistical learning',
        'Literature-validated biology',
        'Combined approach'
    ]
})

print('\nCOMPARISON SUMMARY')
print('=' * 60)
print(summary.to_string(index=False))
print('=' * 60)
best = summary.loc[
    summary['MAE (km)'].idxmin(), 'Method'
]
print(f'\nBest performing: {best}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(
    'PISTES: ML vs Physics vs Hybrid\nAccuracy Comparison',
    fontsize=14, fontweight='bold'
)

methods = ['ML\n(Ridge)', 'Physics\n(EVF)', 'Hybrid']
maes    = [ml_mae, physics_mae, hybrid_mae]
colors  = ['#CC4444', '#1A7A4A', '#1F5C99']

bars = axes[0].bar(
    methods, maes, color=colors,
    alpha=0.85, edgecolor='white', linewidth=1.5
)
axes[0].set_title('Mean Absolute Error (km)', fontweight='bold')
axes[0].set_ylabel('MAE (km)')
axes[0].set_ylim(0, max(maes) * 1.3)
for bar, val in zip(bars, maes):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f'{val:.1f} km',
        ha='center', fontweight='bold', fontsize=11
    )
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Scatter: actual vs predicted
axes[1].scatter(
    actual_distance, ml_distance,
    alpha=0.3, color='#CC4444', s=10, label='ML'
)
axes[1].scatter(
    actual_distance, physics_distance,
    alpha=0.3, color='#1A7A4A', s=10, label='Physics'
)
max_val = max(actual_distance.max(), physics_distance.max())
axes[1].plot(
    [0, max_val], [0, max_val],
    'k--', alpha=0.5, label='Perfect'
)
axes[1].set_xlabel('Actual Movement (km)')
axes[1].set_ylabel('Predicted Movement (km)')
axes[1].set_title('Actual vs Predicted', fontweight='bold')
axes[1].legend()
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(
    '../data/output/method_comparison.png',
    dpi=150, bbox_inches='tight'
)
plt.show()
print('Chart saved ✅')

In [ ]:
print('WIND SENSITIVITY TEST')
print('Same location, different winds')
print('=' * 50)

test_winds = [
    (1.0,   0, 'Very light North'),
    (3.0,   0, 'Light North'),
    (6.0,   0, 'Moderate North'),
    (10.0,  0, 'Strong North'),
    (6.0,  90, 'Moderate East'),
    (6.0, 270, 'Moderate West'),
]

results = []
print(f"\n{'Wind':<25} {'ML (km)':>10} "
      f"{'Physics (km)':>14} {'Hybrid (km)':>12}")
print('-' * 65)

for ws, wd, label in test_winds:
    wu = ws * math.cos(math.radians(wd))
    wv = ws * math.sin(math.radians(wd))

    # ML prediction
    try:
        feat = {
            'wind_u': wu, 'wind_v': wv,
            'wind_speed': ws,
            'wind_direction': wd,
            'humidity': 75
        }
        X_t  = np.array([[feat.get(f, 0) for f in FEATURES]])
        X_ts = scaler.transform(X_t)
        ml_d_lat = float(model_lat.predict(X_ts))
        ml_d_lon = float(model_lon.predict(X_ts))
        ml_km    = math.sqrt(
            (ml_d_lat * 111)**2 + (ml_d_lon * 111)**2
        )
    except Exception:
        ml_km = 0.0

    # Physics prediction
    amp   = 1.0 + 0.65 * 0.05 + 75 / 100 * 0.05
    ph_km = (1.5 + ws * 0.6) * amp * 7

    # Hybrid
    hy_km = ph_km * 0.7 + ml_km * 0.3

    results.append(ph_km)
    print(f'{label:<25} {ml_km:>10.2f} '
          f'{ph_km:>14.2f} {hy_km:>12.2f}')

unique_ph = len(set(round(r, 0) for r in results))
print(f'\nPhysics unique values: {unique_ph}/6')
if unique_ph >= 5:
    print('Wind sensitivity: EXCELLENT ✅')
else:
    print('Wind sensitivity: CHECK ⚠️')

In [ ]:
print('\n' + '=' * 55)
print('RESEARCH CONCLUSION')
print('=' * 55)
print(f"""
ML Model (Ridge Regression):
  MAE: {ml_mae:.2f} km
  Requires: 5,097 training pairs
  R²: ~0.001 (wind explains 0.1%)
  Limitation: Low accuracy due to
  missing livestock/trade data

Physics EVF Model:
  MAE: {physics_mae:.2f} km
  Requires: No training data
  Basis: Peer-reviewed biology
  Punyapornwithaya 2025 (LSD midge)
  Alexandersen 2003 (wind dispersal)

Hybrid Model:
  MAE: {hybrid_mae:.2f} km
  Best of both approaches

Research Finding:
  Physics-based EVF is more
  appropriate for data-scarce
  deployment contexts.

  ML improves when trained on
  livestock movement data.

  Hybrid provides balanced output.
""")
print('=' * 55)

pd.DataFrame({
    'method': ['ML', 'Physics', 'Hybrid'],
    'mae_km': [
        round(ml_mae, 2),
        round(physics_mae, 2),
        round(hybrid_mae, 2)
    ]
}).to_csv(
    '../data/output/accuracy_results.csv',
    index=False
)
print('Results saved ✅')